# Notebook 2 — Facial Recognition Model (Haar Cascade + LBPH)
**IoT-Based Smart Waste Management System — Urban Nigeria**

---

## Overview
This notebook implements the edge-computed facial recognition pipeline described in
Section 4.5 (Algorithm 1) using **OpenCV's Haar Cascade detector** and
**Local Binary Pattern Histogram (LBPH)** recognizer.

### Why LBPH over CNN/FaceNet?
| Property | FaceNet / ArcFace | Haar + LBPH (this work) |
|---|---|---|
| Accuracy | ~99.6% | ~82.6% |
| Inference latency (RPi 4) | ~2,100 ms | ~310 ms |
| GPU required | Yes | No |
| Model size | ~90 MB | ~1 MB |
| Offline operation | No | Yes |

The 82.6% accuracy is sufficient for **deterrence purposes**, with false positives
mitigated by NDPA Section 37 human-in-the-loop review.

### Pipeline (Algorithm 1)
1. Grayscale conversion
2. Histogram equalization
3. Haar Cascade face detection
4. LBPH recognition against encrypted local database
5. Confidence threshold gating → alert


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.metrics import classification_report
import time, warnings
warnings.filterwarnings('ignore')

# OpenCV — install with: pip install opencv-contrib-python
try:
    import cv2
    OPENCV_AVAILABLE = True
    print(f"OpenCV version: {cv2.__version__}")
except ImportError:
    OPENCV_AVAILABLE = False
    print("OpenCV not found. Run: pip install opencv-contrib-python")
    print("Simulation mode will be used for performance metrics.")

np.random.seed(42)


## 1. Pipeline Implementation

In [ ]:
# ── Step 1–3: Image preprocessing ────────────────────────────────────────────
def preprocess_image(img):
    """
    Apply the three preprocessing steps from Algorithm 1:
      Step 1: Grayscale conversion
      Step 2: Resize to 320x240
      Step 3: Histogram equalization (improves contrast under varying illumination)
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if OPENCV_AVAILABLE else img
    else:
        gray = img
    resized = cv2.resize(gray, (320, 240)) if OPENCV_AVAILABLE else gray
    equalized = cv2.equalizeHist(resized) if OPENCV_AVAILABLE else resized
    return equalized


# ── Step 4: Haar Cascade face detector ───────────────────────────────────────
def load_haar_cascade():
    """Load the frontal face Haar Cascade classifier from OpenCV data."""
    if not OPENCV_AVAILABLE:
        return None
    cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    cascade = cv2.CascadeClassifier(cascade_path)
    if cascade.empty():
        raise FileNotFoundError(f"Cascade not found at {cascade_path}")
    return cascade

def detect_faces(img, cascade, scale_factor=1.1, min_neighbors=5, min_size=(30,30)):
    """
    Detect faces using Haar Cascade.
    Returns list of (x, y, w, h) bounding boxes.
    """
    if not OPENCV_AVAILABLE or cascade is None:
        return [(50, 50, 120, 120)]   # mock detection for demo
    preprocessed = preprocess_image(img)
    faces = cascade.detectMultiScale(preprocessed,
                                     scaleFactor=scale_factor,
                                     minNeighbors=min_neighbors,
                                     minSize=min_size)
    return faces if len(faces) > 0 else []

print("Preprocessing and detection functions defined.")


In [ ]:
# ── Step 5: LBPH Recognizer ───────────────────────────────────────────────────
class LBPHRecognizer:
    """
    Wrapper around OpenCV's LBPH face recognizer.
    Falls back to a simulated probabilistic model when OpenCV is unavailable.

    Parameters
    ----------
    radius      : int   — radius of the circular LBP operator
    neighbors   : int   — number of sample points on the circle
    grid_x/y    : int   — number of cells in the spatial histogram grid
    tau_match   : float — confidence threshold (lower = more confident in OpenCV)
    """
    def __init__(self, radius=1, neighbors=8, grid_x=8, grid_y=8, tau_match=80.0):
        self.radius    = radius
        self.neighbors = neighbors
        self.grid_x    = grid_x
        self.grid_y    = grid_y
        self.tau_match = tau_match
        self.trained   = False
        self.labels    = {}

        if OPENCV_AVAILABLE:
            self.recognizer = cv2.face.LBPHFaceRecognizer_create(
                radius=radius, neighbors=neighbors,
                grid_x=grid_x, grid_y=grid_y)
        else:
            self.recognizer = None

    def train(self, face_images, label_ids):
        """Train the LBPH model on a set of face images and integer label IDs."""
        if OPENCV_AVAILABLE:
            self.recognizer.train(face_images, np.array(label_ids))
        self.trained = True
        print(f"LBPH trained on {len(face_images)} images, "
              f"{len(set(label_ids))} identities.")

    def predict(self, face_img):
        """
        Predict identity for a single face image.
        Returns (label_id, confidence).
        Lower confidence = better match in OpenCV LBPH convention.
        """
        if OPENCV_AVAILABLE and self.trained:
            t0 = time.perf_counter()
            label, confidence = self.recognizer.predict(face_img)
            latency_ms = (time.perf_counter() - t0) * 1000
            return label, confidence, latency_ms
        else:
            # Simulated prediction: random confidence around realistic values
            confidence = np.random.normal(loc=60, scale=25)
            confidence = np.clip(confidence, 0, 150)
            label = np.random.randint(0, 5)
            latency_ms = np.random.normal(loc=310, scale=21)
            return label, confidence, latency_ms

    def is_match(self, confidence):
        """Returns True if confidence is below threshold (OpenCV convention)."""
        return confidence < self.tau_match

print("LBPHRecognizer class defined.")


## 2. Simulated Performance Evaluation by Illumination Condition

In [ ]:
# ── Simulate recognition performance across illumination conditions ───────────
# Based on design targets from Table 6 of the paper.
# In physical deployment these parameters are measured empirically.

CONDITIONS = {
    'Optimal (>500 lux)':     {'recall': 0.826, 'precision': 0.883, 'latency_mean': 295, 'latency_std': 18},
    'Indoor (100-500 lux)':   {'recall': 0.781, 'precision': 0.847, 'latency_mean': 308, 'latency_std': 20},
    'Low-light (10-100 lux)': {'recall': 0.684, 'precision': 0.752, 'latency_mean': 325, 'latency_std': 23},
    'Night (<10 lux)':        {'recall': 0.412, 'precision': 0.581, 'latency_mean': 348, 'latency_std': 28},
}

N_TRIALS = 100   # simulated trials per condition

results = {}
for condition, params in CONDITIONS.items():
    recalls, precisions, latencies = [], [], []
    for _ in range(N_TRIALS):
        # Simulate recall/precision with noise around design target
        r = np.clip(np.random.normal(params['recall'],    0.04), 0, 1)
        p = np.clip(np.random.normal(params['precision'], 0.04), 0, 1)
        l = np.random.normal(params['latency_mean'], params['latency_std'])
        recalls.append(r); precisions.append(p); latencies.append(l)
    results[condition] = {
        'recall_mean':    np.mean(recalls),
        'recall_std':     np.std(recalls),
        'precision_mean': np.mean(precisions),
        'precision_std':  np.std(precisions),
        'latency_mean':   np.mean(latencies),
        'latency_std':    np.std(latencies),
        'f1_mean': 2*np.mean(recalls)*np.mean(precisions)/(np.mean(recalls)+np.mean(precisions))
    }

print(f"{'Condition':<28} {'Recall':>8} {'Precision':>10} {'F1':>8} {'Latency (ms)':>14}")
print("-" * 72)
for cond, r in results.items():
    print(f"{cond:<28} {r['recall_mean']*100:>7.1f}% "
          f"{r['precision_mean']*100:>9.1f}% "
          f"{r['f1_mean']*100:>7.1f}% "
          f"{r['latency_mean']:>12.1f}")


## 3. Visualisation — Performance by Illumination

In [ ]:
# ── Performance bar charts ────────────────────────────────────────────────────
labels     = list(results.keys())
recalls    = [r['recall_mean']*100    for r in results.values()]
precisions = [r['precision_mean']*100 for r in results.values()]
latencies  = [r['latency_mean']       for r in results.values()]
r_std      = [r['recall_std']*100     for r in results.values()]
l_std      = [r['latency_std']        for r in results.values()]

short_labels = ['Optimal
>500 lux', 'Indoor
100-500', 'Low-light
10-100', 'Night
<10 lux']
x = np.arange(len(labels))
w = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Recall & Precision
bars1 = ax1.bar(x - w/2, recalls,    w, label='Recall',    color='#1A5276', alpha=0.85, yerr=r_std, capsize=4)
bars2 = ax1.bar(x + w/2, precisions, w, label='Precision', color='#2ECC71', alpha=0.85, capsize=4)
ax1.axhline(82.6, color='#E74C3C', lw=1.5, linestyle='--', label='Design target recall (82.6%)')
ax1.set_xticks(x); ax1.set_xticklabels(short_labels, fontsize=9)
ax1.set_ylabel('Performance (%)', fontsize=11)
ax1.set_ylim(30, 105)
ax1.set_title('Recognition Performance by Illumination', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3, axis='y')

# Latency
ax2.bar(x, latencies, color='#8E44AD', alpha=0.8, yerr=l_std, capsize=4)
ax2.axhline(310, color='#E74C3C', lw=1.5, linestyle='--', label='Design target (310 ms)')
ax2.set_xticks(x); ax2.set_xticklabels(short_labels, fontsize=9)
ax2.set_ylabel('Inference Latency (ms)', fontsize=11)
ax2.set_title('Edge Inference Latency by Illumination', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('facial_recognition_performance.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Full Pipeline — Live Usage Skeleton

In [ ]:
# ── Full pipeline skeleton (Algorithm 1) ─────────────────────────────────────
# This is the production code structure to deploy on Raspberry Pi.
# Replace 'capture_image_from_esp32cam()' with your actual camera interface.

def run_recognition_pipeline(force_delta, lid_state, dt_lid,
                              image, cascade, recognizer,
                              theta_f=0.5, tau_match=80.0, lid_gate=5.0):
    """
    Full event-triggered anomaly detection and recognition pipeline.
    Implements Algorithm 1 from the paper.

    Parameters
    ----------
    force_delta : float — ΔW from FSR reading (kg)
    lid_state   : int   — 1 = lid open, 0 = lid closed
    dt_lid      : float — seconds since last lid-open event
    image       : array — captured image from ESP32-CAM
    cascade     : cv2.CascadeClassifier
    recognizer  : LBPHRecognizer
    theta_f     : float — anomaly force threshold (kg)
    tau_match   : float — LBPH confidence threshold (lower = stricter)
    lid_gate    : float — minimum dt_lid to qualify as anomaly (s)

    Returns
    -------
    alert       : bool
    evidence    : dict with timestamp, face_id, confidence, latency_ms
    fill_level  : float — current fill level percentage
    """
    import datetime

    alert    = False
    evidence = {}

    # ── Step 1: Check anomaly gating conditions (Eq. 5) ──────────────────────
    cond_force = force_delta > theta_f
    cond_lid   = lid_state == 0
    cond_time  = dt_lid > lid_gate

    if cond_force and cond_lid and cond_time:
        # ── Step 2: Capture and preprocess image ──────────────────────────────
        preprocessed = preprocess_image(image)

        # ── Step 3: Detect faces ──────────────────────────────────────────────
        faces = detect_faces(image, cascade)

        for (x, y, w, h) in faces:
            face_roi = preprocessed[y:y+h, x:x+w] if OPENCV_AVAILABLE else preprocessed

            # ── Step 4: LBPH recognition ──────────────────────────────────────
            label, confidence, latency_ms = recognizer.predict(face_roi)

            if recognizer.is_match(confidence):
                alert = True
                evidence = {
                    'timestamp':  datetime.datetime.now().isoformat(),
                    'face_id':    label,
                    'confidence': confidence,
                    'latency_ms': latency_ms,
                    'action':     'FLAGGED_FOR_HUMAN_REVIEW',  # NDPA s.37
                }
                print(f"ALERT | ID={label} | conf={confidence:.1f} | "
                      f"latency={latency_ms:.0f}ms")

    return alert, evidence

print("Full pipeline skeleton defined.")
print("Deploy on Raspberry Pi 4 by replacing image input with ESP32-CAM stream.")
